# Hugging Face Ecosystem Walkthrough

This notebook recreates the code cells shown in the Hugging Face presentation. It is organized cell by cell, with short markdown notes before each code block so the workflow can be run and understood in order.

The main libraries used are `transformers`, `datasets`, `tokenizers`, `huggingface_hub`, and `diffusers`.

## Cell 1: Environment setup

Install the Hugging Face ecosystem libraries and PyTorch. The extra packages `huggingface_hub`, `diffusers`, and `accelerate` are included because later cells use Hub authentication and Stable Diffusion pipelines.

In [2]:
# Cell 1: Environment Setup in Google Colab
# Using the exclamation mark or %pip executes installation commands inside the notebook.
%pip install -q transformers datasets tokenizers torch torchvision torchaudio huggingface_hub diffusers accelerate

# Verification of installation and system check
import transformers
import datasets
import tokenizers
import torch

print(f"Transformers Version: {transformers.__version__}")
print(f"PyTorch CUDA Available: {torch.cuda.is_available()}")


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip
c:\Users\Radhakrishna\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.0.post2)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(


Note: you may need to restart the kernel to use updated packages.
Transformers Version: 4.57.3
PyTorch CUDA Available: True


In [1]:
import torch

print(torch.cuda.is_available())
print(torch.version.cuda)

if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

True
12.8
NVIDIA GeForce RTX 4060 Laptop GPU


## Cell 2: Inspect BERT tensor shapes

This cell downloads `bert-base-uncased`, tokenizes a short input sentence, runs a forward pass, and prints the final hidden-state tensor shape. For BERT, the final shape is usually `[batch_size, sequence_length, hidden_size]`.

In [3]:
# Cell 2: Downloading and instantiating BERT to inspect tensor architectures
from transformers import AutoTokenizer, AutoModel
import torch

# Load the base uncased BERT model and tokenizer
model_name = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name)

# Prepare input text and extract tensor representations
input_text = "hello hugging face"
inputs = tokenizer(input_text, return_tensors="pt")

# Forward pass through the model without gradient calculation
with torch.no_grad():
    outputs = model(**inputs)

# Inspect the final hidden state tensor dimensions
last_hidden_state = outputs.last_hidden_state
print(f"Tokenized Sequence: {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])}")
print(f"Hidden State Tensor Shape: [Batch, Seq Len, Hidden_Dim] -> {list(last_hidden_state.shape)}")

Tokenized Sequence: ['[CLS]', 'hello', 'hugging', 'face', '[SEP]']
Hidden State Tensor Shape: [Batch, Seq Len, Hidden_Dim] -> [1, 5, 768]


## Cell 3: Hugging Face Hub authentication

Public models can be downloaded without authentication. Gated/private models and write operations need an access token. Keep tokens in environment variables or Colab Secrets, never hardcode real credentials in a notebook.

In [4]:
# Cell 3: Handling Authentication for Gated Models and Private Hub Access
from huggingface_hub import login, whoami
import os

# Best Practice: Use environment variables or Colab Secrets, avoid hardcoding strings.
# Here we simulate the process if an API key was required.
HF_ACCESS_TOKEN = os.environ.get("HF_TOKEN", "YOUR_HF_ACCESS_TOKEN_HERE")

try:
    if HF_ACCESS_TOKEN == "YOUR_HF_ACCESS_TOKEN_HERE":
        raise ValueError("No Hugging Face token found in HF_TOKEN.")

    # Programmatic login to the Hugging Face Hub
    login(token=HF_ACCESS_TOKEN, add_to_git_credential=True)
    user_info = whoami()
    print(f"Successfully authenticated as: {user_info['name']}")
except Exception as e:
    print("Authentication skipped or failed. Falling back to public open-source model access.")
    print("Note: Public models (BERT, T5, etc.) do not require authentication.")
    print(f"Reason: {e}")

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


Successfully authenticated as: RK0297


## Cell 4: Load and inspect a dataset

The `datasets` library downloads datasets from the Hub and exposes them as split dictionaries. Here, the IMDB sentiment dataset has `train` and `test` splits with binary labels.

In [5]:
# Cell 4: Loading and Inspecting the IMDb Dataset Dictionary
from datasets import load_dataset

# Streamlined downloading and caching of the IMDB dataset from the Hub
print("Downloading IMDb dataset from Hugging Face Hub...")
imdb_dataset = load_dataset("imdb")

# Print the overall architecture of the Dataset Dictionary
print("\n--- Dataset Dictionary Structure ---")
print(imdb_dataset)

# Inspect a specific sample from the training split
print("\n--- Training Split Sample Extract ---")
sample_review = imdb_dataset["train"][0]
print(f"Review Text: {sample_review['text'][:150]}...")
print(f"Assigned Label: {sample_review['label']} (0 = Negative, 1 = Positive)")


--- Dataset Dictionary Structure ---
DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

--- Training Split Sample Extract ---
Review Text: I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard th...
Assigned Label: 0 (0 = Negative, 1 = Positive)


## Cell 5: Pair a tokenizer with its model

A tokenizer and model must come from the same checkpoint family. The tokenizer creates IDs using the checkpoint vocabulary, and the model expects those IDs to match its learned embeddings.

In [6]:
# Cell 5: Manual Instantiation of NLP Lifecycles via Auto Classes
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Define our target architecture from the public Hub
checkpoint_name = "distilbert-base-uncased-finetuned-sst-2-english"

# Step 1: Load the Tokenizer (handles vocabulary mapping)
print(f"Fetching Tokenizer: {checkpoint_name}...")
tokenizer = AutoTokenizer.from_pretrained(checkpoint_name)

# Step 2: Load the Neural Model (handles mathematical weights and architecture)
print(f"Fetching Model Weights: {checkpoint_name}...")
model = AutoModelForSequenceClassification.from_pretrained(checkpoint_name)

print("\nLifecycle components successfully synchronized and loaded into memory.")
print(f"Model Configuration parameters: {model.config.num_labels} classification labels detected.")

Fetching Tokenizer: distilbert-base-uncased-finetuned-sst-2-english...
Fetching Model Weights: distilbert-base-uncased-finetuned-sst-2-english...

Lifecycle components successfully synchronized and loaded into memory.
Model Configuration parameters: 2 classification labels detected.


## Cell 6: Text classification for spam-like examples

This example uses a text-classification pipeline and then maps model labels into a simple operational decision. In a production spam detector, this mapping should be trained and evaluated on a real spam dataset.

In [ ]:
# Cell 6: Text Classification Pipeline for Spam Detection
from transformers import pipeline

# FIXED: Swapped to a highly reliable, active spam classification model
classifier = pipeline(
    "text-classification",
    model="skandavivek2/spam-classifier"
)

emails = [
    "Congratulations! You have won a 500 INR Amazon gift card. Click here to claim now.",
    "Hi Amit, let's have a meeting tomorrow at 12 p.m. regarding the project deployment.",
    "URGENT: Your Gmail account has been compromised. Provide your password immediately."
]

# Standard execution
predictions = classifier(emails)

print("--- Spam Detection Engine ---")
for email, pred in zip(emails, predictions):
    # FIX: Match the exact uppercase string your model is outputting
    is_spam = "SPAM" if pred["label"].upper() == "SPAM" else "NOT SPAM"

    print(f"Email: {email[:50]}...")
    print(f"Raw Label: {pred['label']} | Score: {pred['score']:.2f} -> Classified As: {is_spam}\n")

Device set to use cuda:0


--- Spam Detection Engine ---
Email: Congratulations! You have won a 500 INR Amazon gif...
Raw Label: SPAM | Score: 1.00 -> Classified As: SPAM

Email: Hi Amit, let's have a meeting tomorrow at 12 p.m. ...
Raw Label: HAM | Score: 1.00 -> Classified As: NOT SPAM

Email: URGENT: Your Gmail account has been compromised. P...
Raw Label: SPAM | Score: 1.00 -> Classified As: SPAM



## Cell 7: Translation with Helsinki-NLP

This example demonstrates how to use an out-of-the-box translation model. Here we use Helsinki-NLP's English-to-Spanish translation model.

In [ ]:
# Cell 7: Translation Pipeline using Helsinki-NLP/opus-mt-en-es
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Switch to a dedicated Translation model for English-to-Spanish
model_id = "Helsinki-NLP/opus-mt-en-es"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id)

input_string = "My name is Amit Diwan and I love cricket."

# No prefix needed here
input_ids = tokenizer(input_string, return_tensors="pt").input_ids
outputs = model.generate(input_ids, max_length=50)

translated_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print(f"Source: {input_string}")
print(f"Translation Output: {translated_text}")

tokenizer_config.json:   0%|          | 0.00/44.0 [00:00<?, ?B/s]

c:\Users\Radhakrishna\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Radhakrishna\.cache\huggingface\hub\models--Helsinki-NLP--opus-mt-en-es. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config.json: 0.00B [00:00, ?B/s]

source.spm:   0%|          | 0.00/802k [00:00<?, ?B/s]

target.spm:   0%|          | 0.00/826k [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/312M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/293 [00:00<?, ?B/s]

Source: My name is Amit Diwan and I love cricket.
Translation Output: Me llamo Amit Diwan y me encanta el cricket.


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


model.safetensors:   0%|          | 0.00/312M [00:00<?, ?B/s]